# LLM Uncertainty Benchmark — Google Colab
Runs the full CP benchmark (5 models × 5 tasks, n=50) using Ollama on Colab.

**Runtime:** GPU is not required — Ollama runs on CPU. Use a standard Colab instance.

**Time estimate:** ~2–3 hours for all models × all tasks.

## 1. Install Ollama

In [ ]:
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama server in background
import subprocess, time
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
print('Ollama server started')

## 2. Clone repo and install dependencies

In [ ]:
!git clone https://github.com/YOUR_USERNAME/LLM-Uncertainty-Study.git
%cd LLM-Uncertainty-Study
!pip install -q numpy scikit-learn tqdm requests matplotlib

## 3. Pull models
Pick the models you want. Comment out any you don't need to save time.

In [ ]:
MODELS = [
    'qwen3.5:2b',
    'qwen3.5:4b',
    'qwen3.5:9b',   # slow on CPU
    'gemma4:e4b',
    'llama3.1:latest',
]

for model in MODELS:
    print(f'Pulling {model}...')
    !ollama pull {model}

## 4. Configuration

In [ ]:
SAMPLES  = 50       # n=50
PROMPT   = 'base'
ICL      = 'icl1'
ALPHA    = 0.1

DATASETS = [
    'mmlu_10k',
    'cosmosqa_10k',
    'hellaswag_10k',
    'halu_dialogue',
    'halu_summarization',
]

import os
os.makedirs('outputs_base', exist_ok=True)
os.makedirs('figures', exist_ok=True)
print('Config OK')

## 5. Generate logits (all models × all datasets)

In [ ]:
for model in MODELS:
    for dataset in DATASETS:
        print(f'\n--- {model} | {dataset} ---')
        !python utils/generate_logits.py \
            --model         {model} \
            --file          {dataset}.json \
            --prompt_method {PROMPT} \
            --few_shot      1 \
            --max_samples   {SAMPLES} \
            --output_dir    outputs_base

## 6. Run CP evaluation

In [ ]:
datasets_arg = ' '.join(DATASETS)

for model in MODELS:
    print(f'\n=== Evaluating {model} ===')
    !python main.py \
        --model          {model} \
        --data_names     {datasets_arg} \
        --prompt_methods {PROMPT} \
        --icl_methods    {ICL} \
        --max_samples    {SAMPLES} \
        --alpha          {ALPHA}

## 7. Generate figures

In [ ]:
!python plot_results.py --samples {SAMPLES} --prompt {PROMPT} --icl {ICL}

## 8. Display figures

In [ ]:
from IPython.display import Image, display
import glob

for png in sorted(glob.glob('figures/*.png')):
    print(f'\n{png}')
    display(Image(png))

## 9. Print summary table

In [ ]:
import json, numpy as np

key = f'{PROMPT}_{ICL}'

print(f"{'Model':<18} ", end='')
for d in DATASETS:
    label = d.replace('_10k','').replace('halu_','')
    print(f"{label:>18}", end='')
print()
print(f"{'':18} ", end='')
for _ in DATASETS:
    print(f"{'CR%  Acc%  SS':>18}", end='')
print()
print('-' * (18 + 18 * len(DATASETS)))

for model in MODELS:
    path = f'outputs_base/{model}_all_results.json'
    if not os.path.exists(path):
        print(f'{model:<18}  (no results)')
        continue
    res = json.load(open(path))
    print(f'{model:<18} ', end='')
    for d in DATASETS:
        if d not in res:
            print(f"{'N/A':>18}", end='')
            continue
        acc = 100 * res[d]['Acc'][key]
        cr  = 100 * np.mean([res[d]['LAC_coverage'][key], res[d]['APS_coverage'][key]])
        ss  =       np.mean([res[d]['LAC_set_size'][key],  res[d]['APS_set_size'][key]])
        print(f"{cr:5.1f} {acc:5.1f} {ss:4.2f}  ", end='')
    print()

## 10. Download results

In [ ]:
# Zip and download all results + figures
!zip -r benchmark_results.zip outputs_base/ figures/

from google.colab import files
files.download('benchmark_results.zip')